# 06B - Unified Domain/Speaker Ablation SER

Notebook này gộp các file 06B trước đây vào một file duy nhất:

- `06B_Domain_Speaker_Ablation_SER.ipynb`: bản chạy từ đầu.
- `06B_part2_resume_from_06_advanced_save.ipynb`: bản resume từ Kaggle dataset chứa cache/report.
- `06B_part2_continue_from_extracted_outputs.ipynb`: bản continue từ output đã giải nén.
- `06b-part2-resume-from-06-advanced-save.ipynb`: bản trùng chức năng với resume.

Notebook unified hỗ trợ 3 chế độ:

```text
1. Fresh run:
   IMPORT_PREVIOUS_RUN_OUTPUTS=0
   PART2_REQUIRE_RESTORED_CACHE=0

2. Resume from Kaggle dataset/cache:
   IMPORT_PREVIOUS_RUN_OUTPUTS=1
   PART2_REQUIRE_RESTORED_CACHE=1
   SAVED_OUTPUT_DATASET=06_advanced_save

3. Continue from extracted local/Kaggle output:
   IMPORT_PREVIOUS_RUN_OUTPUTS=1
   PREVIOUS_RUN_OUTPUT_ROOT=/path/to/06_Advanced_Model_outputs
   PART2_REQUIRE_RESTORED_CACHE=0 hoặc 1 tùy có cache hay không
```

Nếu tìm thấy cache cũ, notebook đọc cache từ đó. Nếu không tìm thấy cache và `PART2_REQUIRE_RESTORED_CACHE=0`, notebook sẽ tự trích xuất feature/cache lại từ `ser_processed`.


## Paper References and Why They Are Used

### 1. Domain-Adversarial Training

**Ganin et al., 2016 - Domain-Adversarial Training of Neural Networks**  
Link: https://arxiv.org/abs/1505.07818

Cách làm của paper:

- Feature extractor tạo embedding `z`.
- Label classifier học task chính.
- Domain classifier cố đoán domain.
- Gradient Reversal Layer (GRL) đảo gradient từ domain loss về feature extractor.
- Kết quả mong muốn: embedding vẫn phân loại tốt task chính nhưng khó đoán domain.

Áp dụng trong notebook:

```text
z_fused -> emotion head
z_fused -> GRL -> domain head
```

Domain ở đây là dataset/corpus: RAVDESS, CREMA-D, TESS, SAVEE.

### 2. Domain Adversarial for Acoustic Emotion Recognition

**Abdelwahab & Busso, 2018 - Domain Adversarial for Acoustic Emotion Recognition**  
Link: https://arxiv.org/abs/1804.07690

Cách làm của paper:

- Áp dụng adversarial multitask learning cho SER.
- Mục tiêu là giảm mismatch giữa train/test domain trong acoustic emotion recognition.

Áp dụng trong notebook:

- So sánh `B0_no_adversarial` và `B1_domain_adv`.
- Nếu random giảm nhẹ nhưng strict tăng hoặc domain accuracy giảm, đó là dấu hiệu domain adversarial có ích.

### 3. Speaker-Invariant SER

**Tu et al., 2019 - Towards adversarial learning of speaker-invariant representation for SER**  
Link: https://arxiv.org/abs/1903.09606

**Li et al., 2019 - Speaker-invariant Affective Representation Learning via Adversarial Training**  
Link: https://arxiv.org/abs/1911.01533

Cách làm chung:

- Thêm speaker classifier để đoán speaker identity từ embedding.
- Dùng adversarial learning/GRL để ép embedding bớt chứa speaker identity.
- Mục tiêu là cải thiện speaker-independent setting.

Áp dụng trong notebook:

```text
S1: speaker probe only
    z_fused.detach() -> speaker head
    speaker head học đo speaker, encoder không bị ảnh hưởng.

S2: speaker adversarial
    z_fused -> GRL -> speaker head
    encoder bị ép làm speaker khó đoán hơn.
```

Lưu ý: speaker cue và emotion cue có thể chồng nhau, ví dụ pitch, energy, timbre. Vì vậy speaker adversarial dùng lambda nhỏ và warm-up.

### 4. Supervised Contrastive Learning

**Khosla et al., 2020 - Supervised Contrastive Learning**  
Link: https://arxiv.org/abs/2004.11362  
Repo: https://github.com/HobbitLong/SupContrast

**Xiang, 2024 - Cross-Corpus SER Based on Supervised Contrastive Learning**  
Link: https://arxiv.org/abs/2411.19803

Cách làm:

- Embedding cùng class được kéo gần nhau.
- Embedding khác class được đẩy xa nhau.
- Hữu ích khi cùng emotion biểu hiện khác nhau theo speaker/dataset.

Áp dụng trong notebook:

```text
z_fused -> SupCon projection head -> normalized embedding
L_total += alpha * L_supcon
```

Notebook dùng SupCon nhẹ (`SUPCON_WEIGHT=0.05`) để tránh phá classifier.

### 5. WavLM and Multi-Representation SER

**Chen et al., 2021 - WavLM**  
Link: https://arxiv.org/abs/2110.13900  
Official repo: https://github.com/microsoft/unilm/tree/master/wavlm

Notebook giữ hướng WavLM frozen + adapter vì:

- hợp Kaggle GPU;
- cache embedding được;
- ít overfit hơn full fine-tune trên dataset SER nhỏ;
- bổ sung representation học từ speech-scale pretraining.

### 6. SE Attention, SpecAugment, mixup

- SENet / SE block: https://arxiv.org/abs/1709.01507
- SpecAugment: https://arxiv.org/abs/1904.08779
- mixup: https://arxiv.org/abs/1710.09412

Áp dụng:

- Spectrogram branch dùng SE/channel attention để học channel nào quan trọng.
- SpecAugment mask time/frequency trên train spectrogram.
- mixup trộn input/label để regularize, nhưng được tắt trong batch SupCon để tránh positive-pair label bị mơ hồ.

## Công Thức Chính

### 1. Emotion Classification

Với embedding hợp nhất:

$$
z = F(x)
$$

Emotion head:

$$
p_y = \operatorname{softmax}(C_y(z))
$$

Cross entropy:

$$
\mathcal{L}_{emo}
= -\sum_{k=1}^{K} y_k \log p_{y,k}
$$

### 2. Gradient Reversal Layer

GRL giữ nguyên tensor ở forward:

$$
\operatorname{GRL}(z) = z
$$

Nhưng đảo gradient ở backward:

$$
\frac{\partial \operatorname{GRL}(z)}{\partial z}
= -\lambda I
$$

Domain adversarial:

$$
p_d = C_d(\operatorname{GRL}(z))
$$

Speaker adversarial:

$$
p_s = C_s(\operatorname{GRL}(z))
$$

### 3. Speaker Probe

Trong config `S1_speaker_probe`, speaker head chỉ đo leakage:

$$
p_s = C_s(\operatorname{stopgrad}(z))
$$

Speaker head học đo speaker, nhưng encoder không nhận gradient từ speaker loss. Nếu speaker accuracy cao, embedding còn chứa speaker identity.

### 4. Supervised Contrastive Loss

Normalize projection:

$$
v_i = \frac{g(z_i)}{\|g(z_i)\|_2}
$$

Positive set:

$$
P(i)=\{p \ne i \mid y_p = y_i\}
$$

SupCon:

$$
\mathcal{L}_{supcon}
= \sum_i
-\frac{1}{|P(i)|}
\sum_{p \in P(i)}
\log
\frac{\exp(v_i^\top v_p / \tau)}
{\sum_{a \ne i}\exp(v_i^\top v_a / \tau)}
$$

Trong notebook, SupCon chỉ bật ở `C1_domain_speaker_adv_supcon`.

### 5. Total Loss

Tổng quát:

$$
\mathcal{L}
= \mathcal{L}_{emo}
+ \beta_d \mathcal{L}_{domain}
+ \beta_s \mathcal{L}_{speaker}
+ \alpha \mathcal{L}_{supcon}
$$

Trong đó:

- `B0`: chỉ có `L_emo`.
- `B1`: thêm `L_domain` qua GRL.
- `S1`: thêm speaker probe, nhưng dùng `stopgrad(z)`.
- `S2`: thêm `L_speaker` qua GRL.
- `D3`: domain + speaker adversarial.
- `C1`: domain + speaker adversarial + SupCon nhẹ.

### 6. Stacking Deep + RBF-SVM

Deep model tạo:

$$
p_{deep}
$$

RBF-SVM trên statistical features tạo:

$$
p_{svm}
$$

Meta-stacking học trên validation probabilities:

$$
p_{final}
= C_{meta}([p_{deep}; p_{svm}])
$$

Mục tiêu là tận dụng cả representation học sâu và handcrafted statistical features.

## Kiến Trúc Notebook 06B

Kiến trúc nền giữ lại từ notebook 06:

```text
Audio 16 kHz
  |-- Branch A: temporal acoustic sequence
  |      MFCC + delta + delta-delta + RMS/ZCR/spectral features
  |      -> 1D-CNN -> BiLSTM -> attention pooling -> z_temporal
  |
  |-- Branch B: spectrogram branch
  |      log-Mel + delta log-Mel + delta-delta log-Mel
  |      -> 2D-CNN + ResidualSE blocks -> z_spectral
  |
  |-- Branch C: pretrained speech branch
  |      raw waveform -> frozen WavLM-base-plus -> adapter MLP -> z_wavlm
  |
  |-- Branch D: statistical branch
         handcrafted statistical vector -> StatsMLP -> z_stats
         handcrafted statistical vector -> RBF-SVM -> p_svm

Fusion:
  z = FusionMLP(concat[z_temporal, z_spectral, z_wavlm, z_stats])

Heads:
  emotion head -> p_deep
  optional domain head + GRL
  optional speaker head/probe + GRL or stopgrad
  optional SupCon projection head

Final:
  meta-stacking([p_deep, p_svm]) -> p_final -> 6 emotion labels
```

6 labels:

```text
neutral, happy, sad, angry, fear, disgust
```

Điểm mới của 06B:

1. Có `speaker_y` từ `speaker_id`.
2. Có `speaker_head`.
3. Có chế độ speaker probe để đo leakage.
4. Có speaker adversarial qua GRL.
5. Có SupCon projection head.
6. Tự chạy 6 config x 2 protocol và xuất bảng so sánh.

In [ ]:
import os
import json
import time
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, balanced_accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_MULTI_GPU = os.getenv("USE_MULTI_GPU", "0") == "1"
torch.backends.cudnn.benchmark = False

def maybe_data_parallel(model, name="model"):
    if USE_MULTI_GPU and GPU_COUNT > 1:
        print(f"Using DataParallel for {name} on {GPU_COUNT} GPUs")
        return nn.DataParallel(model)
    return model

print("Device:", DEVICE)
print("GPU_COUNT:", GPU_COUNT, "USE_MULTI_GPU:", USE_MULTI_GPU)

## 4. Configuration

Kaggle quick test:

```text
QUICK_RUN=1
RUN_PRETRAINED_SPEECH=0
MAX_EPOCHS=1
RUN_SINGLE_DATASET=0
```

Kaggle full test:

```text
QUICK_RUN=0
RUN_PRETRAINED_SPEECH=1
PRETRAINED_SPEECH_MODEL=microsoft/wavlm-base-plus
MAX_EPOCHS=35
USE_DOMAIN_ADVERSARIAL=1
ADV_LAMBDA_MAX=0.05
USE_MULTI_GPU=1
```


In [ ]:
QUICK_RUN = os.getenv("QUICK_RUN", "0") == "1"
QUICK_RUN_PER_GROUP = int(os.getenv("QUICK_RUN_PER_GROUP", "18"))

TARGET_SR = 16_000
TARGET_DURATION = float(os.getenv("TARGET_DURATION", "3.0"))
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)
N_FFT = int(os.getenv("N_FFT", "400"))
WIN_LENGTH = int(os.getenv("WIN_LENGTH", "400"))
HOP_LENGTH = int(os.getenv("HOP_LENGTH", "160"))
N_MFCC = int(os.getenv("N_MFCC", "40"))
N_MELS = int(os.getenv("N_MELS", "96"))
MAX_FRAMES = int(1 + TARGET_LENGTH // HOP_LENGTH)

COMMON_EMOTIONS = ["neutral", "happy", "sad", "angry", "fear", "disgust"]
LABEL_TO_ID = {label: i for i, label in enumerate(COMMON_EMOTIONS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))
MAX_EPOCHS = int(os.getenv("MAX_EPOCHS", "35"))
PATIENCE = int(os.getenv("PATIENCE", "5"))
LR = float(os.getenv("LR", "1e-3"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "3e-4"))
DROPOUT = float(os.getenv("DROPOUT", "0.35"))
LABEL_SMOOTHING = float(os.getenv("LABEL_SMOOTHING", "0.06"))

RUN_COMBINED_RANDOM = os.getenv("RUN_COMBINED_RANDOM", "1") == "1"
RUN_COMBINED_STRICT = os.getenv("RUN_COMBINED_STRICT", "1") == "1"
RUN_SINGLE_DATASET = os.getenv("RUN_SINGLE_DATASET", "0") == "1"

RUN_PRETRAINED_SPEECH = os.getenv("RUN_PRETRAINED_SPEECH", "1") == "1"
PRETRAINED_SPEECH_MODEL = os.getenv("PRETRAINED_SPEECH_MODEL", "microsoft/wavlm-base-plus")
SPEECH_EMBED_BATCH_SIZE = int(os.getenv("SPEECH_EMBED_BATCH_SIZE", "16" if GPU_COUNT > 1 else "8"))

USE_AUGMENTATION = os.getenv("USE_AUGMENTATION", "1") == "1"
USE_WAVEFORM_AUGMENTATION = os.getenv("USE_WAVEFORM_AUGMENTATION", "0") == "1"
WAVEFORM_AUGMENT_COPIES = int(os.getenv("WAVEFORM_AUGMENT_COPIES", "0"))
TEMPORAL_WAVE_AUG_PROB = float(os.getenv("TEMPORAL_WAVE_AUG_PROB", "0.45"))
SPECTRAL_AUG_PROB = float(os.getenv("SPECTRAL_AUG_PROB", "0.45"))
MIXUP_ALPHA = float(os.getenv("MIXUP_ALPHA", "0.10"))
MIXUP_PROB = float(os.getenv("MIXUP_PROB", "0.20"))

USE_CLASS_WEIGHTS = os.getenv("USE_CLASS_WEIGHTS", "1") == "1"
USE_BALANCED_SAMPLER = os.getenv("USE_BALANCED_SAMPLER", "1") == "1"
CACHE_FEATURES = os.getenv("CACHE_FEATURES", "1") == "1"
ENABLE_RICH_PITCH = os.getenv("ENABLE_RICH_PITCH", "1") == "1"
STATS_PCA_COMPONENTS = int(os.getenv("STATS_PCA_COMPONENTS", "256"))

ADV_WARMUP_EPOCHS = int(os.getenv("ADV_WARMUP_EPOCHS", "5"))
DEFAULT_DOMAIN_LAMBDA_MAX = float(os.getenv("DEFAULT_DOMAIN_LAMBDA_MAX", "0.05"))
DEFAULT_SPEAKER_LAMBDA_MAX = float(os.getenv("DEFAULT_SPEAKER_LAMBDA_MAX", "0.03"))
DEFAULT_DOMAIN_LOSS_WEIGHT = float(os.getenv("DEFAULT_DOMAIN_LOSS_WEIGHT", "0.50"))
DEFAULT_SPEAKER_LOSS_WEIGHT = float(os.getenv("DEFAULT_SPEAKER_LOSS_WEIGHT", "0.25"))
SUPCON_WEIGHT = float(os.getenv("SUPCON_WEIGHT", "0.05"))
SUPCON_TEMPERATURE = float(os.getenv("SUPCON_TEMPERATURE", "0.10"))
RUN_CONFIGS = [x.strip() for x in os.getenv("RUN_CONFIGS", "D3_domain_speaker_adv").split(",") if x.strip()]
RUN_SVM_BASELINE = os.getenv("RUN_SVM_BASELINE", "1") == "1"

print({
    "QUICK_RUN": QUICK_RUN,
    "MAX_EPOCHS": MAX_EPOCHS,
    "RUN_PRETRAINED_SPEECH": RUN_PRETRAINED_SPEECH,
    "PRETRAINED_SPEECH_MODEL": PRETRAINED_SPEECH_MODEL,
    "RUN_CONFIGS": RUN_CONFIGS,
    "DEFAULT_DOMAIN_LAMBDA_MAX": DEFAULT_DOMAIN_LAMBDA_MAX,
    "DEFAULT_SPEAKER_LAMBDA_MAX": DEFAULT_SPEAKER_LAMBDA_MAX,
    "DEFAULT_DOMAIN_LOSS_WEIGHT": DEFAULT_DOMAIN_LOSS_WEIGHT,
    "DEFAULT_SPEAKER_LOSS_WEIGHT": DEFAULT_SPEAKER_LOSS_WEIGHT,
    "SUPCON_WEIGHT": SUPCON_WEIGHT,
    "USE_WAVEFORM_AUGMENTATION": USE_WAVEFORM_AUGMENTATION,
})

In [ ]:
def find_ser_processed():
    candidates = []
    env_path = os.getenv("SER_PROCESSED", "").strip()
    if env_path:
        candidates.append(Path(env_path))
    candidates.extend([
        Path("/kaggle/input/datasets/quanghuy225/ser-processed/ser_processed"),
        Path("/kaggle/input/ser-processed/ser_processed"),
        Path("/kaggle/input/ser-processed"),
        Path("/kaggle/working/ser_processed"),
        Path.cwd() / "ser_processed",
        Path.cwd().parent / "ser_processed",
        Path.cwd().parent / "01&02_Data_and_DataProcessing" / "ser_processed",
        Path("D:/UTE/Speech Programming/Speech Project/01&02_Data_and_DataProcessing/ser_processed"),
    ])
    for candidate in candidates:
        if (candidate / "metadata.csv").exists():
            return candidate.resolve()
    for root in [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]:
        if root.exists():
            for metadata_path in root.rglob("metadata.csv"):
                if metadata_path.parent.name == "ser_processed":
                    return metadata_path.parent.resolve()
    raise FileNotFoundError("Cannot find ser_processed/metadata.csv")

SER_PROCESSED = find_ser_processed()
AUDIO_16K_DIR = SER_PROCESSED / "audio_16k"
PROJECT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / "06_Advanced_Model_outputs"
REPORT_DIR = OUTPUT_DIR / "reports"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
PRED_DIR = OUTPUT_DIR / "predictions"
CACHE_DIR = OUTPUT_DIR / "cache"
for d in [REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("SER_PROCESSED:", SER_PROCESSED)
print("OUTPUT_DIR:", OUTPUT_DIR)


## Import Previous D3 Run Outputs

Cell nay tim output cu co `06B_ablation_metrics.csv` va `06B_ablation_history.csv`, sau do copy `reports/`, `figures/`, `predictions/` vao `OUTPUT_DIR` hien tai. Nho do notebook tiep tuc duoc tu phan visualize/save ma khong train lai run da xong.

In [ ]:

IMPORT_PREVIOUS_RUN_OUTPUTS = os.getenv("IMPORT_PREVIOUS_RUN_OUTPUTS", "1") == "1"
PREVIOUS_RUN_OUTPUT_ROOT = os.getenv("PREVIOUS_RUN_OUTPUT_ROOT", "").strip()

def report_dir_has_06b_metrics(report_dir):
    return (
        report_dir.exists()
        and (report_dir / "06B_ablation_metrics.csv").exists()
        and (report_dir / "06B_ablation_history.csv").exists()
    )

def find_previous_run_output_root():
    candidates = []
    if PREVIOUS_RUN_OUTPUT_ROOT:
        candidates.append(Path(PREVIOUS_RUN_OUTPUT_ROOT))

    # Local workspace path after extracting Kaggle output.
    candidates.extend([
        Path.cwd() / "06_Advanced_Model" / "outputs" / "06B_part2_D3_outputs" / "06_Advanced_Model_outputs",
        Path.cwd() / "outputs" / "06B_part2_D3_outputs" / "06_Advanced_Model_outputs",
        Path.cwd() / "06B_part2_D3_outputs" / "06_Advanced_Model_outputs",
        Path.cwd() / "06_Advanced_Model_outputs",
        PROJECT_ROOT / "06_Advanced_Model_outputs",
    ])

    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(input_root.rglob("06_Advanced_Model_outputs"))
        candidates.extend(input_root.rglob("reports"))
        candidates.extend(input_root.rglob("report"))

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.exists():
            continue
        seen.add(candidate)
        possible_roots = [candidate]
        if candidate.name in ["reports", "report"]:
            possible_roots.append(candidate.parent)
        for root in possible_roots:
            for report_dir in [root / "reports", root / "report", root]:
                if report_dir_has_06b_metrics(report_dir):
                    return report_dir.parent if report_dir.name in ["reports", "report"] else report_dir
    return None

def copy_tree_files(src_dir, dst_dir, patterns=("*",)):
    if src_dir is None or not src_dir.exists():
        return 0
    dst_dir.mkdir(parents=True, exist_ok=True)
    copied = 0
    for pattern in patterns:
        for src in src_dir.rglob(pattern):
            if not src.is_file():
                continue
            dst = dst_dir / src.relative_to(src_dir)
            dst.parent.mkdir(parents=True, exist_ok=True)
            if (not dst.exists()) or src.stat().st_size != dst.stat().st_size:
                shutil.copy2(src, dst)
                copied += 1
    return copied

PREVIOUS_OUTPUT_ROOT = find_previous_run_output_root() if IMPORT_PREVIOUS_RUN_OUTPUTS else None
print("PREVIOUS_OUTPUT_ROOT:", PREVIOUS_OUTPUT_ROOT)
if PREVIOUS_OUTPUT_ROOT is not None:
    prev_reports = PREVIOUS_OUTPUT_ROOT / "reports"
    if not prev_reports.exists():
        prev_reports = PREVIOUS_OUTPUT_ROOT / "report"
    prev_figures = PREVIOUS_OUTPUT_ROOT / "figures"
    prev_predictions = PREVIOUS_OUTPUT_ROOT / "predictions"

    copied_reports = copy_tree_files(prev_reports, REPORT_DIR, patterns=("*.csv", "*.json"))
    copied_figures = copy_tree_files(prev_figures, FIGURE_DIR, patterns=("*.png", "*.jpg", "*.jpeg", "*.pdf"))
    copied_predictions = copy_tree_files(prev_predictions, PRED_DIR, patterns=("*.csv",))
    print("Copied previous outputs:",
          {"reports": copied_reports, "figures": copied_figures, "predictions": copied_predictions})
    print("Metrics target exists:", (REPORT_DIR / "06B_ablation_metrics.csv").exists())
    print("History target exists:", (REPORT_DIR / "06B_ablation_history.csv").exists())
else:
    print("No previous 06B metrics/history found. Notebook will train requested configs.")


## Optional Saved Cache/Report Locator

Cell này thay thế các notebook part2 cũ. Nó tự tìm dataset/cache cũ trong `/kaggle/input`, local output hoặc biến `SAVED_OUTPUT_ROOT`.

- Nếu tìm thấy cache: `CACHE_DIR` sẽ trỏ tới cache cũ để tiết kiệm thời gian.
- Nếu không tìm thấy cache và `PART2_REQUIRE_RESTORED_CACHE=0`: notebook giữ `CACHE_DIR` trong output hiện tại và tự build cache.
- Nếu không tìm thấy cache và `PART2_REQUIRE_RESTORED_CACHE=1`: notebook dừng sớm để tránh chạy nhầm.


In [ ]:
PART2_REQUIRE_RESTORED_CACHE = os.getenv("PART2_REQUIRE_RESTORED_CACHE", "0") == "1"
SAVED_OUTPUT_DATASET = os.getenv("SAVED_OUTPUT_DATASET", "06_advanced_save")

def normalize_slug(s):
    return str(s).lower().replace("-", "_").replace(" ", "_")

def cache_dir_has_npz(cache_dir):
    return cache_dir.exists() and cache_dir.is_dir() and any(cache_dir.glob("*.npz"))

def find_saved_output_root():
    env_root = os.getenv("SAVED_OUTPUT_ROOT", "").strip()
    candidates = []
    if env_root:
        candidates.append(Path(env_root))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        wanted = normalize_slug(SAVED_OUTPUT_DATASET)
        for p in input_root.iterdir():
            if p.is_dir():
                name = normalize_slug(p.name)
                if wanted in name or name in wanted or ("advanced" in name and "save" in name):
                    candidates.append(p)
        candidates.extend(input_root.rglob("06_Advanced_Model_outputs"))

    candidates.extend([
        PROJECT_ROOT / "06_Advanced_Model_outputs",
        Path.cwd() / "06_Advanced_Model_outputs",
        Path.cwd() / "06_Advanced_Model" / "outputs" / "06B_part2_D3_outputs" / "06_Advanced_Model_outputs",
    ])

    seen = set()
    for root in candidates:
        root = root.resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)
        possible_cache_dirs = [
            root / "cache",
            root / "06_Advanced_Model_outputs" / "cache",
            root,
        ]
        if any(cache_dir_has_npz(d) for d in possible_cache_dirs):
            return root

    if input_root.exists():
        for cache_dir in input_root.rglob("cache"):
            if cache_dir_has_npz(cache_dir):
                return cache_dir.parent.resolve()

    return None

SAVED_OUTPUT_ROOT = find_saved_output_root()
SAVED_CACHE_DIR = None
SAVED_REPORT_DIR = None

if SAVED_OUTPUT_ROOT is not None:
    if cache_dir_has_npz(SAVED_OUTPUT_ROOT / "cache"):
        SAVED_CACHE_DIR = SAVED_OUTPUT_ROOT / "cache"
    elif cache_dir_has_npz(SAVED_OUTPUT_ROOT / "06_Advanced_Model_outputs" / "cache"):
        SAVED_CACHE_DIR = SAVED_OUTPUT_ROOT / "06_Advanced_Model_outputs" / "cache"
    elif cache_dir_has_npz(SAVED_OUTPUT_ROOT):
        SAVED_CACHE_DIR = SAVED_OUTPUT_ROOT

    if (SAVED_OUTPUT_ROOT / "reports").exists():
        SAVED_REPORT_DIR = SAVED_OUTPUT_ROOT / "reports"
    elif (SAVED_OUTPUT_ROOT / "report").exists():
        SAVED_REPORT_DIR = SAVED_OUTPUT_ROOT / "report"
    elif (SAVED_OUTPUT_ROOT / "06_Advanced_Model_outputs" / "reports").exists():
        SAVED_REPORT_DIR = SAVED_OUTPUT_ROOT / "06_Advanced_Model_outputs" / "reports"

if SAVED_CACHE_DIR is not None:
    CACHE_DIR = SAVED_CACHE_DIR
    print("Using restored cache.")
else:
    print("No restored cache found. Using current OUTPUT cache directory.")
    if PART2_REQUIRE_RESTORED_CACHE:
        raise FileNotFoundError(
            "PART2_REQUIRE_RESTORED_CACHE=1 but no saved cache was found. "
            "Set SAVED_OUTPUT_ROOT or add the Kaggle dataset containing cache/."
        )

print("SAVED_OUTPUT_ROOT:", SAVED_OUTPUT_ROOT)
print("READ/WRITE CACHE_DIR:", CACHE_DIR)
print("OLD REPORT DIR:", SAVED_REPORT_DIR)
print("NEW OUTPUT_DIR:", OUTPUT_DIR)
if CACHE_DIR.exists():
    print("Cache files:")
    for p in sorted(CACHE_DIR.glob("*")):
        if p.is_file():
            print(" -", p.name, f"({p.stat().st_size/1024**2:.1f} MB)")
if SAVED_REPORT_DIR is not None:
    print("Old report files:")
    for p in sorted(SAVED_REPORT_DIR.glob("*")):
        if p.is_file():
            print(" -", p.name)


## 5. Metadata and Protocols

Notebook giữ ba nhóm protocol:

- `combined_random`: gần với nhiều paper/repo dùng random split.
- `combined_strict_no_tess`: speaker-aware hơn, loại TESS vì chỉ có 2 speaker.
- `single-dataset`: train/test riêng từng corpus để so với paper thường báo theo từng dataset.

Điểm quan trọng: augmentation chỉ được dùng trong train loader hoặc train index, validation/test luôn dùng feature gốc.


In [ ]:
metadata = pd.read_csv(SER_PROCESSED / "metadata.csv")
metadata = metadata[metadata["emotion"].isin(COMMON_EMOTIONS)].copy()
if "readable" in metadata.columns:
    metadata = metadata[metadata["readable"] == True].copy()

metadata["label_id"] = metadata["emotion"].map(LABEL_TO_ID).astype(int)
metadata["speaker_id"] = metadata["speaker_id"].astype(str)
metadata["domain_id"] = metadata["dataset"].astype("category").cat.codes.astype(int)
metadata["speaker_code"] = metadata["speaker_id"].astype("category").cat.codes.astype(int)
DOMAIN_NAMES = list(metadata["dataset"].astype("category").cat.categories)
SPEAKER_NAMES = list(metadata["speaker_id"].astype("category").cat.categories)
NUM_DOMAINS = len(DOMAIN_NAMES)
NUM_SPEAKERS = len(SPEAKER_NAMES)

if QUICK_RUN:
    metadata = (
        metadata.sort_values(["dataset", "emotion", "sample_id"])
        .groupby(["dataset", "emotion"], group_keys=False)
        .head(QUICK_RUN_PER_GROUP)
        .reset_index(drop=True)
    )
    metadata["domain_id"] = metadata["dataset"].astype("category").cat.codes.astype(int)
    metadata["speaker_code"] = metadata["speaker_id"].astype("category").cat.codes.astype(int)
    DOMAIN_NAMES = list(metadata["dataset"].astype("category").cat.categories)
    SPEAKER_NAMES = list(metadata["speaker_id"].astype("category").cat.categories)
    NUM_DOMAINS = len(DOMAIN_NAMES)
    NUM_SPEAKERS = len(SPEAKER_NAMES)
else:
    metadata = metadata.reset_index(drop=True)

metadata["split_strict_original"] = metadata["split"].astype(str).str.lower() if "split" in metadata.columns else "train"
metadata.loc[~metadata["split_strict_original"].isin(["train", "validation", "test"]), "split_strict_original"] = "train"

metadata["strict_include"] = metadata["dataset"] != "TESS"
metadata["split_strict_project"] = metadata["split_strict_original"]
savee_plan = {"savee_DC": "train", "savee_JE": "train", "savee_JK": "validation", "savee_KL": "test"}
savee_mask = metadata["dataset"].eq("SAVEE")
metadata.loc[savee_mask, "split_strict_project"] = metadata.loc[savee_mask, "speaker_id"].map(savee_plan).fillna("train")
metadata.loc[metadata["dataset"].eq("TESS"), "split_strict_project"] = "excluded"

train_idx, temp_idx = train_test_split(
    metadata.index,
    test_size=0.30,
    random_state=SEED,
    stratify=metadata["label_id"],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=metadata.loc[temp_idx, "label_id"],
)
metadata["split_random"] = "train"
metadata.loc[val_idx, "split_random"] = "validation"
metadata.loc[test_idx, "split_random"] = "test"

display(metadata.groupby(["dataset", "emotion"]).size().unstack(fill_value=0))
display(metadata.groupby(["dataset", "split_strict_project"]).size().unstack(fill_value=0))

metadata.groupby(["dataset", "emotion"]).size().reset_index(name="count").to_csv(
    REPORT_DIR / "dataset_emotion_distribution.csv", index=False
)
metadata.groupby(["dataset", "split_strict_project"]).size().reset_index(name="count").to_csv(
    REPORT_DIR / "project_strict_split_by_dataset.csv", index=False
)


speaker_y = metadata["speaker_code"].to_numpy().astype(np.int64)
print("NUM_DOMAINS:", NUM_DOMAINS, DOMAIN_NAMES)
print("NUM_SPEAKERS:", NUM_SPEAKERS)

## Optional Cache Check

Notebook unified có thể chạy theo hai kiểu:

- Có cache cũ: kiểm tra file cache và dùng luôn.
- Không có cache cũ: nếu `PART2_REQUIRE_RESTORED_CACHE=0`, notebook sẽ tự build cache ở các cell sau.


In [ ]:
expected_feature_cache = CACHE_DIR / (
    f"advanced_features_mfcc{N_MFCC}_mel{N_MELS}_"
    f"{int(TARGET_DURATION)}s_{TARGET_SR}hz_pitch{int(ENABLE_RICH_PITCH)}_{len(metadata)}files.npz"
)
expected_aug_cache = CACHE_DIR / (
    f"advanced_wave_aug_copies{WAVEFORM_AUGMENT_COPIES}_"
    f"mfcc{N_MFCC}_mel{N_MELS}_{int(TARGET_DURATION)}s_{len(metadata)}files.npz"
)
expected_speech_cache = CACHE_DIR / (
    "speech_embed_" + PRETRAINED_SPEECH_MODEL.replace("/", "_") + f"_{int(TARGET_DURATION)}s_{len(metadata)}files.npz"
)

expected = [expected_feature_cache]
if USE_WAVEFORM_AUGMENTATION and WAVEFORM_AUGMENT_COPIES > 0:
    expected.append(expected_aug_cache)
if RUN_PRETRAINED_SPEECH:
    expected.append(expected_speech_cache)

missing = [p for p in expected if not p.exists()]
print("Expected cache files:")
for p in expected:
    print(("OK      " if p.exists() else "MISSING "), p)

if missing and PART2_REQUIRE_RESTORED_CACHE:
    raise FileNotFoundError(
        "Missing required cache while PART2_REQUIRE_RESTORED_CACHE=1. Missing: "
        + ", ".join(str(p) for p in missing)
    )
elif missing:
    print("Some cache files are missing, but PART2_REQUIRE_RESTORED_CACHE=0, so notebook will build them if needed.")


## 6. Feature Extraction

### Branch A temporal sequence

\[
X_{temporal} \in \mathbb{R}^{N \times T \times 132}
\]

Gồm:

```text
40 MFCC + 40 delta + 40 delta-delta
+ RMS + ZCR + centroid + bandwidth + rolloff + 7 spectral contrast
= 132 features/frame
```

### Branch B spectral tensor

\[
X_{spectral} \in \mathbb{R}^{N \times 3 \times M \times T}
\]

3 channel:

```text
log-Mel
delta log-Mel
delta-delta log-Mel
```

### Branch D statistical vector

Tính nhiều thống kê theo thời gian:

```text
mean, std, min, max, median, p25, p75, IQR, range, skewness, kurtosis
```


In [ ]:
def center_crop_or_pad(y, target_length=TARGET_LENGTH):
    if len(y) > target_length:
        start = (len(y) - target_length) // 2
        y = y[start:start + target_length]
    elif len(y) < target_length:
        pad = target_length - len(y)
        y = np.pad(y, (pad // 2, pad - pad // 2), mode="constant")
    return y.astype(np.float32)

def normalize_audio(y):
    y = np.asarray(y, dtype=np.float32)
    y = np.nan_to_num(y)
    peak = np.max(np.abs(y)) + 1e-8
    if peak > 1.0:
        y = y / peak
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def read_audio(row):
    cached = AUDIO_16K_DIR / f"{row.sample_id}.wav"
    if cached.exists():
        y, sr = sf.read(cached)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != TARGET_SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
    else:
        y, sr = librosa.load(row.filepath, sr=TARGET_SR, mono=True)
    return normalize_audio(center_crop_or_pad(y, TARGET_LENGTH))

def random_waveform_augment(y):
    if not USE_WAVEFORM_AUGMENTATION:
        return y.astype(np.float32)
    y_aug = y.astype(np.float32).copy()
    if random.random() < 0.35:
        try:
            y_aug = librosa.effects.pitch_shift(y_aug, sr=TARGET_SR, n_steps=random.uniform(-2.0, 2.0))
        except Exception:
            pass
    if random.random() < 0.30:
        try:
            y_aug = librosa.effects.time_stretch(y_aug, rate=random.uniform(0.88, 1.12))
            y_aug = center_crop_or_pad(y_aug, TARGET_LENGTH)
        except Exception:
            pass
    if random.random() < 0.45:
        rms = np.sqrt(np.mean(y_aug ** 2) + 1e-8)
        snr_db = random.uniform(14.0, 28.0)
        noise_rms = rms / (10 ** (snr_db / 20.0))
        y_aug = y_aug + np.random.normal(0.0, noise_rms, size=len(y_aug)).astype(np.float32)
    if random.random() < 0.35:
        y_aug *= random.uniform(0.72, 1.28)
    if random.random() < 0.35:
        shift = random.randint(-int(0.18 * TARGET_SR), int(0.18 * TARGET_SR))
        y_aug = np.roll(y_aug, shift)
        if shift > 0:
            y_aug[:shift] = 0
        elif shift < 0:
            y_aug[shift:] = 0
    if random.random() < 0.20:
        width = random.randint(int(0.035 * TARGET_SR), int(0.14 * TARGET_SR))
        start = random.randint(0, max(0, len(y_aug) - width))
        y_aug[start:start + width] *= random.uniform(0.0, 0.15)
    return normalize_audio(center_crop_or_pad(y_aug, TARGET_LENGTH))

def pad_time(x, target, axis=-1):
    if x.shape[axis] > target:
        sl = [slice(None)] * x.ndim
        sl[axis] = slice(0, target)
        return x[tuple(sl)]
    if x.shape[axis] < target:
        pad_width = [(0, 0)] * x.ndim
        pad_width[axis] = (0, target - x.shape[axis])
        return np.pad(x, pad_width, mode="constant")
    return x

def safe_skew_kurtosis(x, axis=1):
    mean = x.mean(axis=axis, keepdims=True)
    std = x.std(axis=axis, keepdims=True) + 1e-6
    z = (x - mean) / std
    skew = np.mean(z ** 3, axis=axis)
    kurt = np.mean(z ** 4, axis=axis) - 3.0
    return skew, kurt

def stats_full(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 1:
        x = x[None, :]
    mean = x.mean(axis=1)
    std = x.std(axis=1)
    mn = x.min(axis=1)
    mx = x.max(axis=1)
    med = np.median(x, axis=1)
    p25 = np.percentile(x, 25, axis=1)
    p75 = np.percentile(x, 75, axis=1)
    iqr = p75 - p25
    rg = mx - mn
    skew, kurt = safe_skew_kurtosis(x, axis=1)
    return np.concatenate([mean, std, mn, mx, med, p25, p75, iqr, rg, skew, kurt]).astype(np.float32)

def extract_pitch_stats(y):
    if not ENABLE_RICH_PITCH:
        return np.zeros(11, dtype=np.float32)
    try:
        f0 = librosa.yin(y, fmin=50, fmax=500, sr=TARGET_SR, frame_length=max(1024, N_FFT), hop_length=HOP_LENGTH)
        f0 = f0[np.isfinite(f0)]
        f0 = f0[(f0 >= 50) & (f0 <= 500)]
        if len(f0) < 3:
            return np.zeros(11, dtype=np.float32)
        return stats_full(f0).astype(np.float32)
    except Exception:
        return np.zeros(11, dtype=np.float32)

def frame_entropy(power):
    p = power / (np.sum(power, axis=0, keepdims=True) + 1e-8)
    return (-np.sum(p * np.log(p + 1e-8), axis=0, keepdims=True)).astype(np.float32)

def extract_representations(row=None, y_override=None):
    y_audio = read_audio(row) if y_override is None else normalize_audio(center_crop_or_pad(y_override, TARGET_LENGTH))

    mfcc = librosa.feature.mfcc(y=y_audio, sr=TARGET_SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    rms = librosa.feature.rms(y=y_audio, frame_length=WIN_LENGTH, hop_length=HOP_LENGTH)
    zcr = librosa.feature.zero_crossing_rate(y_audio, frame_length=WIN_LENGTH, hop_length=HOP_LENGTH)
    centroid = librosa.feature.spectral_centroid(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    bandwidth = librosa.feature.spectral_bandwidth(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    rolloff = librosa.feature.spectral_rolloff(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    contrast = librosa.feature.spectral_contrast(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_bands=6)

    mel = librosa.feature.melspectrogram(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, n_mels=N_MELS, power=2.0)
    logmel = librosa.power_to_db(mel, ref=np.max)
    d_logmel = librosa.feature.delta(logmel)
    d2_logmel = librosa.feature.delta(logmel, order=2)

    chroma_stft = librosa.feature.chroma_stft(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    chroma_cqt = librosa.feature.chroma_cqt(y=y_audio, sr=TARGET_SR, hop_length=HOP_LENGTH)
    chroma_cens = librosa.feature.chroma_cens(y=y_audio, sr=TARGET_SR, hop_length=HOP_LENGTH)
    power = np.abs(librosa.stft(y_audio, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)) ** 2
    energy = power.sum(axis=0, keepdims=True)
    entropy = frame_entropy(power)

    temporal = np.concatenate([
        pad_time(mfcc, MAX_FRAMES, axis=1),
        pad_time(delta, MAX_FRAMES, axis=1),
        pad_time(delta2, MAX_FRAMES, axis=1),
        pad_time(rms, MAX_FRAMES, axis=1),
        pad_time(zcr, MAX_FRAMES, axis=1),
        pad_time(centroid, MAX_FRAMES, axis=1),
        pad_time(bandwidth, MAX_FRAMES, axis=1),
        pad_time(rolloff, MAX_FRAMES, axis=1),
        pad_time(contrast, MAX_FRAMES, axis=1),
    ], axis=0).T.astype(np.float32)

    spectral = np.stack([
        pad_time(logmel, MAX_FRAMES, axis=1),
        pad_time(d_logmel, MAX_FRAMES, axis=1),
        pad_time(d2_logmel, MAX_FRAMES, axis=1),
    ], axis=0).astype(np.float32)

    stat_vec = np.concatenate([
        stats_full(mfcc), stats_full(delta), stats_full(delta2),
        stats_full(chroma_stft), stats_full(chroma_cqt), stats_full(chroma_cens),
        stats_full(centroid), stats_full(bandwidth), stats_full(rolloff), stats_full(contrast),
        stats_full(rms), stats_full(zcr), stats_full(energy), stats_full(entropy), stats_full(logmel),
        extract_pitch_stats(y_audio),
    ]).astype(np.float32)

    return temporal, spectral, stat_vec, y_audio


In [ ]:

import zipfile

USE_MEMMAP_CACHE = os.getenv("USE_MEMMAP_CACHE", "1") == "1"
MEMMAP_CACHE_DIR = PROJECT_ROOT / "06B_memmap_cache"
MEMMAP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

cache_name = (
    f"advanced_features_mfcc{N_MFCC}_mel{N_MELS}_"
    f"{int(TARGET_DURATION)}s_{TARGET_SR}hz_pitch{int(ENABLE_RICH_PITCH)}_{len(metadata)}files.npz"
)
cache_path = CACHE_DIR / cache_name

def extract_npz_member_to_npy(npz_path, member_name, out_dir):
    out_path = out_dir / member_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(npz_path, "r") as zf:
        info = zf.getinfo(member_name)
        if out_path.exists() and out_path.stat().st_size == info.file_size:
            print("Memmap member exists:", out_path.name, f"({out_path.stat().st_size/1024**2:.1f} MB)")
            return out_path
        print("Extracting for memmap:", member_name, "->", out_path)
        with zf.open(member_name) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst, length=1024 * 1024 * 64)
    return out_path

if CACHE_FEATURES and cache_path.exists():
    if USE_MEMMAP_CACHE:
        temporal_npy = extract_npz_member_to_npy(cache_path, "X_temporal.npy", MEMMAP_CACHE_DIR)
        spectral_npy = extract_npz_member_to_npy(cache_path, "X_spectral.npy", MEMMAP_CACHE_DIR)
        X_temporal = np.load(temporal_npy, mmap_mode="r")
        X_spectral = np.load(spectral_npy, mmap_mode="r")
        with np.load(cache_path) as data:
            X_stats = data["X_stats"].astype(np.float32)
            y = data["y"].astype(np.int64)
            domain_y = data["domain_y"].astype(np.int64)
        print("Loaded feature cache with memmap:", cache_path)
    else:
        with np.load(cache_path) as data:
            X_temporal = data["X_temporal"].astype(np.float32)
            X_spectral = data["X_spectral"].astype(np.float32)
            X_stats = data["X_stats"].astype(np.float32)
            y = data["y"].astype(np.int64)
            domain_y = data["domain_y"].astype(np.int64)
        print("Loaded feature cache into RAM:", cache_path)
else:
    X_temporal, X_spectral, X_stats = [], [], []
    start = time.time()
    for i, row in enumerate(metadata.itertuples(index=False)):
        a, b, c, _ = extract_representations(row)
        X_temporal.append(a)
        X_spectral.append(b)
        X_stats.append(c)
        if (i + 1) % 250 == 0:
            print(f"Extracted {i+1}/{len(metadata)} files in {(time.time()-start)/60:.1f} min")
    X_temporal = np.stack(X_temporal).astype(np.float32)
    X_spectral = np.stack(X_spectral).astype(np.float32)
    X_stats = np.stack(X_stats).astype(np.float32)
    y = metadata["label_id"].to_numpy(np.int64)
    domain_y = metadata["domain_id"].to_numpy(np.int64)
    if CACHE_FEATURES:
        np.savez_compressed(cache_path, X_temporal=X_temporal, X_spectral=X_spectral, X_stats=X_stats, y=y, domain_y=domain_y)
        print("Saved feature cache:", cache_path)

TEMPORAL_DIM = X_temporal.shape[-1]
STATS_DIM = X_stats.shape[-1]
print("X_temporal:", X_temporal.shape, type(X_temporal))
print("X_spectral:", X_spectral.shape, type(X_spectral))
print("X_stats:", X_stats.shape)
print("domain names:", DOMAIN_NAMES)


## 7. Train-Only Waveform Augmented Cache

Cache này có thể được tạo cho toàn metadata để tái sử dụng, nhưng khi train chỉ lấy index thuộc `train_idx`. Validation/test không dùng augmented feature.


In [ ]:
if USE_WAVEFORM_AUGMENTATION and WAVEFORM_AUGMENT_COPIES > 0:
    aug_cache_name = (
        f"advanced_wave_aug_copies{WAVEFORM_AUGMENT_COPIES}_"
        f"mfcc{N_MFCC}_mel{N_MELS}_{int(TARGET_DURATION)}s_{len(metadata)}files.npz"
    )
    aug_cache_path = CACHE_DIR / aug_cache_name
    if CACHE_FEATURES and aug_cache_path.exists():
        with np.load(aug_cache_path) as aug_data:
            X_temporal_wave_aug = aug_data["X_temporal_wave_aug"].astype(np.float32)
            X_spectral_wave_aug = aug_data["X_spectral_wave_aug"].astype(np.float32)
            X_stats_wave_aug = aug_data["X_stats_wave_aug"].astype(np.float32)
        print("Loaded waveform augmentation cache:", aug_cache_path)
    else:
        aug_temporal, aug_spectral, aug_stats = [], [], []
        start = time.time()
        for copy_i in range(WAVEFORM_AUGMENT_COPIES):
            copy_temporal, copy_spectral, copy_stats = [], [], []
            for i, row in enumerate(metadata.itertuples(index=False)):
                y_base = read_audio(row)
                y_aug = random_waveform_augment(y_base)
                a, b, c, _ = extract_representations(row, y_override=y_aug)
                copy_temporal.append(a)
                copy_spectral.append(b)
                copy_stats.append(c)
                if (i + 1) % 250 == 0:
                    print(f"Wave aug copy {copy_i+1}/{WAVEFORM_AUGMENT_COPIES}: {i+1}/{len(metadata)} in {(time.time()-start)/60:.1f} min")
            aug_temporal.append(np.stack(copy_temporal).astype(np.float32))
            aug_spectral.append(np.stack(copy_spectral).astype(np.float32))
            aug_stats.append(np.stack(copy_stats).astype(np.float32))
        X_temporal_wave_aug = np.stack(aug_temporal).astype(np.float32)
        X_spectral_wave_aug = np.stack(aug_spectral).astype(np.float32)
        X_stats_wave_aug = np.stack(aug_stats).astype(np.float32)
        if CACHE_FEATURES:
            np.savez_compressed(aug_cache_path, X_temporal_wave_aug=X_temporal_wave_aug, X_spectral_wave_aug=X_spectral_wave_aug, X_stats_wave_aug=X_stats_wave_aug)
            print("Saved waveform augmentation cache:", aug_cache_path)
else:
    X_temporal_wave_aug = None
    X_spectral_wave_aug = None
    X_stats_wave_aug = None
    print("Waveform augmentation cache disabled.")

if X_temporal_wave_aug is not None:
    print("X_temporal_wave_aug:", X_temporal_wave_aug.shape)
    print("X_spectral_wave_aug:", X_spectral_wave_aug.shape)
    print("X_stats_wave_aug:", X_stats_wave_aug.shape)


## 8. WavLM Frozen Embeddings

Branch C dùng raw waveform đưa qua WavLM.

Trong notebook:

- WavLM frozen: không fine-tune backbone.
- Mean pooling theo frame để tạo utterance embedding.
- Adapter MLP sẽ học emotion trên embedding đó.


In [ ]:
def extract_pretrained_speech_embeddings():
    if not RUN_PRETRAINED_SPEECH:
        print("Pretrained speech branch disabled.")
        return None
    embed_cache = CACHE_DIR / ("speech_embed_" + PRETRAINED_SPEECH_MODEL.replace("/", "_") + f"_{int(TARGET_DURATION)}s_{len(metadata)}files.npz")
    if CACHE_FEATURES and embed_cache.exists():
        with np.load(embed_cache) as embed_data:
            arr = embed_data["X_speech_embed"].astype(np.float32)
        print("Loaded speech embedding cache:", embed_cache, arr.shape)
        return arr
    try:
        from transformers import AutoFeatureExtractor, AutoModel
    except Exception as e:
        print("Cannot import transformers; skip pretrained speech branch:", e)
        return None
    try:
        feature_extractor = AutoFeatureExtractor.from_pretrained(PRETRAINED_SPEECH_MODEL)
        encoder = AutoModel.from_pretrained(PRETRAINED_SPEECH_MODEL).to(DEVICE)
        encoder = maybe_data_parallel(encoder, name="pretrained speech encoder")
        encoder.eval()
    except Exception as e:
        print("Cannot load pretrained speech model; skip branch:", e)
        return None
    embeddings = []
    start = time.time()
    with torch.no_grad():
        for start_i in range(0, len(metadata), SPEECH_EMBED_BATCH_SIZE):
            rows = metadata.iloc[start_i:start_i + SPEECH_EMBED_BATCH_SIZE]
            waves = [read_audio(row) for row in rows.itertuples(index=False)]
            inputs = feature_extractor(waves, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            out = encoder(**inputs)
            hidden = out.last_hidden_state
            if "attention_mask" in inputs:
                mask = inputs["attention_mask"]
                frame_mask = F.interpolate(mask[:, None, :].float(), size=hidden.shape[1], mode="nearest").squeeze(1)
                pooled = (hidden * frame_mask[..., None]).sum(dim=1) / (frame_mask.sum(dim=1, keepdim=True) + 1e-8)
            else:
                pooled = hidden.mean(dim=1)
            embeddings.append(pooled.cpu().numpy().astype(np.float32))
            done = min(start_i + SPEECH_EMBED_BATCH_SIZE, len(metadata))
            if done % 240 == 0 or done == len(metadata):
                print(f"Speech embeddings {done}/{len(metadata)} in {(time.time()-start)/60:.1f} min")
    X_speech_embed = np.concatenate(embeddings, axis=0).astype(np.float32)
    if CACHE_FEATURES:
        np.savez_compressed(embed_cache, X_speech_embed=X_speech_embed)
        print("Saved speech embedding cache:", embed_cache)
    return X_speech_embed

X_speech_embed = extract_pretrained_speech_embeddings()
if X_speech_embed is not None:
    print("X_speech_embed:", X_speech_embed.shape)


## 9. Dataset, Scaling and Train-Only Augmentation

Scaler luôn fit trên train split:

\[
\mu, \sigma = \mathrm{mean/std}(X_{train})
\]

Validation/test chỉ transform bằng train scaler.

SpecAugment áp dụng cho spectrogram train:

- Time mask: che một khoảng thời gian.
- Frequency mask: che một vùng mel bins.


In [ ]:
def _chunked_sum_sumsq(arr, indices, axes_after_batch, chunk_size=256):
    indices = np.asarray(indices)
    total_sum = None
    total_sumsq = None
    total_count = 0
    for start in range(0, len(indices), chunk_size):
        batch_idx = indices[start:start + chunk_size]
        chunk = np.asarray(arr[batch_idx], dtype=np.float32)
        sum_axes = tuple([0] + [axis + 1 for axis in axes_after_batch])
        chunk_sum = chunk.sum(axis=sum_axes, dtype=np.float64)
        chunk_sumsq = np.square(chunk, dtype=np.float64).sum(axis=sum_axes, dtype=np.float64)
        count = np.prod([chunk.shape[axis + 1] for axis in axes_after_batch]) * chunk.shape[0]
        total_sum = chunk_sum if total_sum is None else total_sum + chunk_sum
        total_sumsq = chunk_sumsq if total_sumsq is None else total_sumsq + chunk_sumsq
        total_count += int(count)
    mean = total_sum / max(1, total_count)
    var = np.maximum(total_sumsq / max(1, total_count) - mean ** 2, 1e-12)
    return mean.astype(np.float32), np.sqrt(var).astype(np.float32)

def compute_scalers(train_idx):
    # Chunked statistics avoid materializing X_temporal[train_idx] or X_spectral[train_idx]
    # in RAM. This is required for Kaggle when the feature cache is memory-mapped.
    temporal_mean, temporal_std = _chunked_sum_sumsq(X_temporal, train_idx, axes_after_batch=(0,), chunk_size=256)
    spectral_mean, spectral_std = _chunked_sum_sumsq(X_spectral, train_idx, axes_after_batch=(1, 2), chunk_size=64)
    scalers = {
        "temporal_mean": temporal_mean.reshape(1, 1, -1).astype(np.float32),
        "temporal_std": (temporal_std.reshape(1, 1, -1) + 1e-6).astype(np.float32),
        "spectral_mean": spectral_mean.reshape(X_spectral.shape[1], 1, 1).astype(np.float32),
        "spectral_std": (spectral_std.reshape(X_spectral.shape[1], 1, 1) + 1e-6).astype(np.float32),
        "stats_scaler": StandardScaler().fit(X_stats[train_idx]),
        "speech_scaler": StandardScaler().fit(X_speech_embed[train_idx]) if X_speech_embed is not None else None,
    }
    return scalers

def augment_temporal(x):
    if not USE_AUGMENTATION:
        return x
    x = x.copy()
    if random.random() < 0.30:
        x = np.roll(x, random.randint(-18, 18), axis=0)
    if random.random() < 0.35:
        x += np.random.normal(0, 0.02, x.shape).astype(np.float32)
    if random.random() < 0.35:
        width = random.randint(8, 36)
        start = random.randint(0, max(0, x.shape[0] - width))
        x[start:start + width, :] = 0
    if random.random() < 0.25:
        width = random.randint(4, 18)
        start = random.randint(0, max(0, x.shape[1] - width))
        x[:, start:start + width] = 0
    return x.astype(np.float32)

def augment_spectral(x):
    if not USE_AUGMENTATION or random.random() > SPECTRAL_AUG_PROB:
        return x
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 4 and x.shape[0] == 1:
        x = x[0]
    if x.ndim != 3:
        raise ValueError(f"augment_spectral expects [C, mel, frames], got {x.shape}")
    x = x.copy()
    _, mels, frames = x.shape
    if random.random() < 0.70:
        width = random.randint(8, min(40, frames))
        start = random.randint(0, max(0, frames - width))
        x[:, :, start:start + width] = 0
    if random.random() < 0.70:
        width = random.randint(4, min(18, mels))
        start = random.randint(0, max(0, mels - width))
        x[:, start:start + width, :] = 0
    if random.random() < 0.30:
        x += np.random.normal(0, 0.02, x.shape).astype(np.float32)
    return x.astype(np.float32)

class MultiBranchDataset(Dataset):
    def __init__(self, indices, scalers, train=False):
        self.indices = np.array(indices)
        self.scalers = scalers
        self.train = train
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        use_wave_aug = self.train and X_temporal_wave_aug is not None and random.random() < TEMPORAL_WAVE_AUG_PROB
        if use_wave_aug:
            copy_i = random.randrange(X_temporal_wave_aug.shape[0])
            temporal_raw = X_temporal_wave_aug[copy_i, i]
            spectral_raw = X_spectral_wave_aug[copy_i, i]
            stats_raw = X_stats_wave_aug[copy_i, i]
        else:
            temporal_raw = X_temporal[i]
            spectral_raw = X_spectral[i]
            stats_raw = X_stats[i]

        temporal = ((temporal_raw - self.scalers["temporal_mean"].squeeze(0)) / self.scalers["temporal_std"].squeeze(0)).astype(np.float32)
        spectral = ((spectral_raw - self.scalers["spectral_mean"]) / self.scalers["spectral_std"]).astype(np.float32)
        if spectral.ndim == 4 and spectral.shape[0] == 1:
            spectral = spectral[0]
        if spectral.ndim != 3:
            raise ValueError(f"spectral sample must be [C, mel, frames], got {spectral.shape}")
        stats = self.scalers["stats_scaler"].transform(stats_raw[None, :]).astype(np.float32)[0]
        if self.train:
            temporal = augment_temporal(temporal)
            spectral = augment_spectral(spectral)
        if X_speech_embed is not None:
            speech = self.scalers["speech_scaler"].transform(X_speech_embed[i:i+1]).astype(np.float32)[0]
        else:
            speech = np.zeros(1, dtype=np.float32)
        return {
            "temporal": torch.from_numpy(temporal),
            "spectral": torch.from_numpy(spectral),
            "stats": torch.from_numpy(stats),
            "speech": torch.from_numpy(speech),
            "label": torch.tensor(y[i], dtype=torch.long),
            "domain": torch.tensor(domain_y[i], dtype=torch.long),
            "speaker": torch.tensor(speaker_y[i], dtype=torch.long),
        }

def make_sampler(indices):
    if not USE_BALANCED_SAMPLER:
        return None
    labels = y[indices]
    datasets = metadata.iloc[indices]["dataset"].to_numpy()
    key_counts = defaultdict(int)
    for ds, lab in zip(datasets, labels):
        key_counts[(ds, int(lab))] += 1
    weights = np.array([1.0 / key_counts[(ds, int(lab))] for ds, lab in zip(datasets, labels)], dtype=np.float32)
    weights = weights / weights.mean()
    return WeightedRandomSampler(torch.from_numpy(weights), num_samples=len(indices), replacement=True)

def make_loader(dataset, indices, train=False):
    sampler = make_sampler(indices) if train else None
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(train and sampler is None), sampler=sampler, num_workers=0, pin_memory=False)

## Model Blocks

Các block dưới đây triển khai đúng kiến trúc 06B: 4 branch representation, fusion MLP, emotion head, domain head, speaker head và SupCon projection head.

In [ ]:
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradientReversalFn.apply(x, lambd)

class AttentionPoolSeq(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1), w.squeeze(-1)

class TemporalCNNBiLSTMBranch(nn.Module):
    def __init__(self, input_dim, emb_dim=192, hidden=128, dropout=DROPOUT):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Conv1d(128, 160, kernel_size=5, padding=4, dilation=2),
            nn.BatchNorm1d(160), nn.GELU(), nn.Dropout(dropout * 0.5),
        )
        self.lstm = nn.LSTM(input_size=160, hidden_size=hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.pool = AttentionPoolSeq(hidden * 2)
        self.proj = nn.Sequential(nn.LayerNorm(hidden * 2), nn.Linear(hidden * 2, emb_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x):
        x = self.norm(x)
        x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(x)
        pooled, _ = self.pool(out)
        return self.proj(pooled)

class SEBlock2D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        mid = max(4, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, mid), nn.GELU(),
            nn.Linear(mid, channels), nn.Sigmoid()
        )
    def forward(self, x):
        w = self.net(x).view(x.size(0), x.size(1), 1, 1)
        return x * w

class ResidualSEBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se = SEBlock2D(out_ch)
        self.skip = nn.Identity() if in_ch == out_ch and stride == 1 else nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.act = nn.GELU()
        self.drop = nn.Dropout2d(dropout * 0.4)
    def forward(self, x):
        out = self.conv(x)
        out = self.se(out)
        out = self.act(out + self.skip(x))
        return self.drop(out)

class SpectralCNNSEBranch(nn.Module):
    def __init__(self, in_ch=3, emb_dim=192, dropout=DROPOUT):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, 48, 5, stride=2, padding=2, bias=False), nn.BatchNorm2d(48), nn.GELU())
        self.blocks = nn.Sequential(
            ResidualSEBlock(48, 64, stride=1, dropout=dropout),
            ResidualSEBlock(64, 96, stride=2, dropout=dropout),
            ResidualSEBlock(96, 128, stride=2, dropout=dropout),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Flatten(), nn.LayerNorm(128), nn.Linear(128, emb_dim), nn.GELU(), nn.Dropout(dropout))
    def forward(self, x):
        return self.proj(self.pool(self.blocks(self.stem(x))))

class SpeechAdapterBranch(nn.Module):
    def __init__(self, input_dim, emb_dim=160, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 384), nn.LayerNorm(384), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(384, emb_dim), nn.GELU(), nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class StatsMLPBranch(nn.Module):
    def __init__(self, input_dim, emb_dim=128, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, emb_dim), nn.GELU(), nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class DomainSpeakerAblationSER(nn.Module):
    def __init__(self, temporal_dim, spectral_channels, stats_dim, speech_dim, num_classes, num_domains, num_speakers):
        super().__init__()
        self.temporal = TemporalCNNBiLSTMBranch(temporal_dim)
        self.spectral = SpectralCNNSEBranch(spectral_channels)
        self.use_speech = speech_dim > 1
        self.speech = SpeechAdapterBranch(speech_dim) if self.use_speech else None
        self.stats = StatsMLPBranch(stats_dim)
        fusion_dim = 192 + 192 + 128 + (160 if self.use_speech else 0)
        self.fusion = nn.Sequential(nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, 256), nn.GELU(), nn.Dropout(DROPOUT))
        self.emotion_head = nn.Linear(256, num_classes)
        self.domain_head = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, num_domains))
        self.speaker_head = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, num_speakers))
        self.supcon_head = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Linear(128, 128))

    def branch_embeddings(self, temporal, spectral, stats, speech):
        parts = [self.temporal(temporal), self.spectral(spectral), self.stats(stats)]
        if self.use_speech:
            parts.append(self.speech(speech))
        return parts

    def forward(self, temporal, spectral, stats, speech, domain_mode="off", speaker_mode="off", domain_lambda=0.0, speaker_lambda=0.0):
        fused = self.fusion(torch.cat(self.branch_embeddings(temporal, spectral, stats, speech), dim=1))
        emo_logits = self.emotion_head(fused)

        if domain_mode == "adv":
            dom_input = grad_reverse(fused, domain_lambda)
        elif domain_mode == "probe":
            dom_input = fused.detach()
        else:
            dom_input = fused.detach()
        dom_logits = self.domain_head(dom_input)

        if speaker_mode == "adv":
            spk_input = grad_reverse(fused, speaker_lambda)
        elif speaker_mode == "probe":
            spk_input = fused.detach()
        else:
            spk_input = fused.detach()
        spk_logits = self.speaker_head(spk_input)

        supcon_z = F.normalize(self.supcon_head(fused), dim=1)
        return emo_logits, dom_logits, spk_logits, fused, supcon_z

## Training Logic for 6 Ablation Configs

Cell này định nghĩa 6 cấu hình từ `bonus.md` và loss tương ứng. `S1_speaker_probe` dùng `stopgrad(z)` để đo speaker leakage mà không làm encoder học speaker mạnh hơn.

In [ ]:
import gc

EXPERIMENT_CONFIGS = {
    "B0_no_adversarial": {
        "domain_mode": "off", "speaker_mode": "off", "use_supcon": False,
        "domain_lambda_max": 0.0, "speaker_lambda_max": 0.0,
        "domain_loss_weight": 0.0, "speaker_loss_weight": 0.0, "supcon_weight": 0.0,
        "note": "Baseline advanced model. No adversarial regularization."
    },
    "B1_domain_adv": {
        "domain_mode": "adv", "speaker_mode": "off", "use_supcon": False,
        "domain_lambda_max": DEFAULT_DOMAIN_LAMBDA_MAX, "speaker_lambda_max": 0.0,
        "domain_loss_weight": DEFAULT_DOMAIN_LOSS_WEIGHT, "speaker_loss_weight": 0.0, "supcon_weight": 0.0,
        "note": "DANN-style domain adversarial training."
    },
    "S1_speaker_probe": {
        "domain_mode": "off", "speaker_mode": "probe", "use_supcon": False,
        "domain_lambda_max": 0.0, "speaker_lambda_max": 0.0,
        "domain_loss_weight": 0.0, "speaker_loss_weight": DEFAULT_SPEAKER_LOSS_WEIGHT, "supcon_weight": 0.0,
        "note": "Speaker head sees stop-gradient fused embedding. Measures speaker leakage without changing encoder."
    },
    "S2_speaker_adv": {
        "domain_mode": "off", "speaker_mode": "adv", "use_supcon": False,
        "domain_lambda_max": 0.0, "speaker_lambda_max": DEFAULT_SPEAKER_LAMBDA_MAX,
        "domain_loss_weight": 0.0, "speaker_loss_weight": DEFAULT_SPEAKER_LOSS_WEIGHT, "supcon_weight": 0.0,
        "note": "Speaker adversarial training with a small lambda."
    },
    "D3_domain_speaker_adv": {
        "domain_mode": "adv", "speaker_mode": "adv", "use_supcon": False,
        "domain_lambda_max": DEFAULT_DOMAIN_LAMBDA_MAX, "speaker_lambda_max": DEFAULT_SPEAKER_LAMBDA_MAX,
        "domain_loss_weight": DEFAULT_DOMAIN_LOSS_WEIGHT, "speaker_loss_weight": DEFAULT_SPEAKER_LOSS_WEIGHT, "supcon_weight": 0.0,
        "note": "Domain + speaker adversarial training."
    },
    "C1_domain_speaker_adv_supcon": {
        "domain_mode": "adv", "speaker_mode": "adv", "use_supcon": True,
        "domain_lambda_max": DEFAULT_DOMAIN_LAMBDA_MAX, "speaker_lambda_max": DEFAULT_SPEAKER_LAMBDA_MAX,
        "domain_loss_weight": DEFAULT_DOMAIN_LOSS_WEIGHT, "speaker_loss_weight": DEFAULT_SPEAKER_LOSS_WEIGHT,
        "supcon_weight": SUPCON_WEIGHT,
        "note": "Domain + speaker adversarial training with a light supervised contrastive loss."
    },
}

ACTIVE_CONFIGS = [name for name in RUN_CONFIGS if name in EXPERIMENT_CONFIGS]
missing_configs = [name for name in RUN_CONFIGS if name not in EXPERIMENT_CONFIGS]
if missing_configs:
    print("Ignoring unknown configs:", missing_configs)
print("ACTIVE_CONFIGS:", ACTIVE_CONFIGS)

def class_weights_for(indices):
    counts = np.bincount(y[indices], minlength=len(COMMON_EMOTIONS)).astype(np.float32)
    weights = counts.sum() / (len(COMMON_EMOTIONS) * np.maximum(counts, 1))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def domain_weights_for(indices):
    counts = np.bincount(domain_y[indices], minlength=NUM_DOMAINS).astype(np.float32)
    weights = counts.sum() / (NUM_DOMAINS * np.maximum(counts, 1))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def speaker_weights_for(indices):
    counts = np.bincount(speaker_y[indices], minlength=NUM_SPEAKERS).astype(np.float32)
    weights = counts.sum() / (NUM_SPEAKERS * np.maximum(counts, 1))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def lambda_for_epoch(epoch, lambda_max):
    if lambda_max <= 0 or epoch <= ADV_WARMUP_EPOCHS:
        return 0.0
    denom = max(1, MAX_EPOCHS - ADV_WARMUP_EPOCHS)
    progress = min(1.0, (epoch - ADV_WARMUP_EPOCHS) / denom)
    # Smooth schedule from DANN-style training.
    return float(lambda_max * (2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0))

def move_batch(batch):
    return {k: v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}

def mixup_tensors(batch, alpha=MIXUP_ALPHA, enabled=True):
    if (not enabled) or alpha <= 0 or random.random() > MIXUP_PROB:
        return batch, batch["label"], None, None
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(batch["label"].size(0), device=batch["label"].device)
    mixed = dict(batch)
    for key in ["temporal", "spectral", "stats", "speech"]:
        mixed[key] = lam * batch[key] + (1 - lam) * batch[key][index]
    return mixed, batch["label"], batch["label"][index], lam

def mixup_ce(logits, y_a, y_b, lam, criterion):
    if y_b is None:
        return criterion(logits, y_a)
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)

def supervised_contrastive_loss(features, labels, temperature=SUPCON_TEMPERATURE):
    if features.size(0) < 2:
        return torch.tensor(0.0, device=features.device)
    labels = labels.contiguous().view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(features.device)
    logits = torch.div(torch.matmul(features, features.T), temperature)
    logits = logits - torch.max(logits, dim=1, keepdim=True)[0].detach()
    logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0), device=features.device)
    mask = mask * logits_mask
    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-8)
    positives = mask.sum(1)
    valid = positives > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=features.device)
    mean_log_prob_pos = (mask * log_prob).sum(1) / torch.clamp(positives, min=1.0)
    return -mean_log_prob_pos[valid].mean()

def run_epoch(model, loader, criteria, optimizer=None, epoch=1, config=None):
    is_train = optimizer is not None
    model.train(is_train)
    config = config or EXPERIMENT_CONFIGS["B0_no_adversarial"]
    domain_lambda = lambda_for_epoch(epoch, config["domain_lambda_max"]) if is_train else 0.0
    speaker_lambda = lambda_for_epoch(epoch, config["speaker_lambda_max"]) if is_train else 0.0

    totals = defaultdict(float)
    all_y, all_pred, all_prob = [], [], []
    all_dom, all_dom_pred = [], []
    all_spk, all_spk_pred = [], []
    start = time.time()

    for batch in loader:
        batch = move_batch(batch)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
            # SupCon uses clean labels; avoid mixup label ambiguity in SupCon config.
            batch_in, y_a, y_b, lam = mixup_tensors(batch, enabled=not config["use_supcon"])
        else:
            batch_in, y_a, y_b, lam = batch, batch["label"], None, None

        with torch.set_grad_enabled(is_train):
            # Pass control values positionally. This avoids a DataParallel scatter edge case
            # where GPU replicas can receive keyword arguments without the tensor inputs.
            emo_logits, dom_logits, spk_logits, fused, supcon_z = model(
                batch_in["temporal"], batch_in["spectral"], batch_in["stats"], batch_in["speech"],
                config["domain_mode"], config["speaker_mode"], domain_lambda, speaker_lambda
            )
            emo_loss = mixup_ce(emo_logits, y_a, y_b, lam, criteria["emotion"])
            domain_loss = criteria["domain"](dom_logits, batch["domain"]) if config["domain_loss_weight"] > 0 else torch.tensor(0.0, device=DEVICE)
            speaker_loss = criteria["speaker"](spk_logits, batch["speaker"]) if config["speaker_loss_weight"] > 0 else torch.tensor(0.0, device=DEVICE)
            supcon_loss = supervised_contrastive_loss(supcon_z, batch["label"]) if (is_train and config["use_supcon"]) else torch.tensor(0.0, device=DEVICE)
            loss = (
                emo_loss
                + config["domain_loss_weight"] * domain_loss
                + config["speaker_loss_weight"] * speaker_loss
                + config["supcon_weight"] * supcon_loss
            )
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        prob = torch.softmax(emo_logits.detach(), dim=1)
        n = len(batch["label"])
        totals["loss"] += loss.item() * n
        totals["emotion_loss"] += emo_loss.item() * n
        totals["domain_loss"] += domain_loss.item() * n
        totals["speaker_loss"] += speaker_loss.item() * n
        totals["supcon_loss"] += supcon_loss.item() * n
        all_y.extend(batch["label"].detach().cpu().numpy())
        all_pred.extend(prob.argmax(dim=1).cpu().numpy())
        all_prob.extend(prob.cpu().numpy())
        all_dom.extend(batch["domain"].detach().cpu().numpy())
        all_dom_pred.extend(dom_logits.detach().argmax(dim=1).cpu().numpy())
        all_spk.extend(batch["speaker"].detach().cpu().numpy())
        all_spk_pred.extend(spk_logits.detach().argmax(dim=1).cpu().numpy())

    all_y = np.array(all_y)
    all_pred = np.array(all_pred)
    all_prob = np.array(all_prob)
    all_dom = np.array(all_dom)
    all_dom_pred = np.array(all_dom_pred)
    all_spk = np.array(all_spk)
    all_spk_pred = np.array(all_spk_pred)
    denom = max(1, len(all_y))
    return {
        "loss": totals["loss"] / denom,
        "emotion_loss": totals["emotion_loss"] / denom,
        "domain_loss": totals["domain_loss"] / denom,
        "speaker_loss": totals["speaker_loss"] / denom,
        "supcon_loss": totals["supcon_loss"] / denom,
        "accuracy": accuracy_score(all_y, all_pred),
        "macro_f1": f1_score(all_y, all_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(all_y, all_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(all_y, all_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(all_y, all_pred, average="macro", zero_division=0),
        "uar": balanced_accuracy_score(all_y, all_pred),
        "domain_accuracy": accuracy_score(all_dom, all_dom_pred) if len(all_dom) else 0.0,
        "speaker_accuracy": accuracy_score(all_spk, all_spk_pred) if len(all_spk) else 0.0,
        "y_true": all_y,
        "y_pred": all_pred,
        "prob": all_prob,
        "elapsed_sec": time.time() - start,
        "domain_lambda": domain_lambda,
        "speaker_lambda": speaker_lambda,
    }

def train_ablation_model(protocol_name, config_name, split_map):
    config = EXPERIMENT_CONFIGS[config_name]
    train_idx, val_idx, test_idx = split_map["train"], split_map["validation"], split_map["test"]
    scalers = compute_scalers(train_idx)
    train_ds = MultiBranchDataset(train_idx, scalers, train=True)
    val_ds = MultiBranchDataset(val_idx, scalers, train=False)
    test_ds = MultiBranchDataset(test_idx, scalers, train=False)
    train_loader = make_loader(train_ds, train_idx, train=True)
    val_loader = make_loader(val_ds, val_idx, train=False)
    test_loader = make_loader(test_ds, test_idx, train=False)

    speech_dim = X_speech_embed.shape[1] if X_speech_embed is not None else 1
    model = DomainSpeakerAblationSER(
        TEMPORAL_DIM, X_spectral.shape[1], STATS_DIM, speech_dim,
        len(COMMON_EMOTIONS), NUM_DOMAINS, NUM_SPEAKERS
    )
    model = maybe_data_parallel(model.to(DEVICE), name=f"{protocol_name}/{config_name}")

    criteria = {
        "emotion": nn.CrossEntropyLoss(weight=class_weights_for(train_idx) if USE_CLASS_WEIGHTS else None, label_smoothing=LABEL_SMOOTHING),
        "domain": nn.CrossEntropyLoss(weight=domain_weights_for(train_idx)),
        "speaker": nn.CrossEntropyLoss(weight=speaker_weights_for(train_idx)),
    }
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    history = []
    best_state, best_epoch, best_val = None, 0, -1.0
    patience_left = PATIENCE
    train_start = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        train_m = run_epoch(model, train_loader, criteria, optimizer=optimizer, epoch=epoch, config=config)
        val_m = run_epoch(model, val_loader, criteria, epoch=epoch, config=config)
        scheduler.step(val_m["macro_f1"])
        row = {
            "protocol": protocol_name,
            "config_name": config_name,
            "epoch": epoch,
            "train_macro_f1": train_m["macro_f1"],
            "val_macro_f1": val_m["macro_f1"],
            "train_loss": train_m["loss"],
            "val_loss": val_m["loss"],
            "train_domain_acc": train_m["domain_accuracy"],
            "val_domain_acc": val_m["domain_accuracy"],
            "train_speaker_acc": train_m["speaker_accuracy"],
            "val_speaker_acc": val_m["speaker_accuracy"],
            "domain_lambda": train_m["domain_lambda"],
            "speaker_lambda": train_m["speaker_lambda"],
            "supcon_loss": train_m["supcon_loss"],
            "lr": optimizer.param_groups[0]["lr"],
            "note": config["note"],
        }
        history.append(row)
        print(
            f"{protocol_name}/{config_name} | epoch {epoch:02d} "
            f"| train f1 {train_m['macro_f1']:.4f} | val f1 {val_m['macro_f1']:.4f} "
            f"| dom acc {train_m['domain_accuracy']:.3f} | spk acc {train_m['speaker_accuracy']:.3f} "
            f"| dlambda {train_m['domain_lambda']:.3f} | slambda {train_m['speaker_lambda']:.3f}"
        )
        if val_m["macro_f1"] > best_val:
            best_val = val_m["macro_f1"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("Early stopping.")
                break

    train_time = time.time() - train_start
    model.load_state_dict(best_state)
    val_final = run_epoch(model, val_loader, criteria, epoch=best_epoch, config=config)
    test_final = run_epoch(model, test_loader, criteria, epoch=best_epoch, config=config)
    result = {
        "history": history,
        "val": val_final,
        "test": test_final,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val,
        "train_time_sec": train_time,
        "config": config,
    }
    model.to("cpu")
    del model, optimizer, scheduler, criteria, train_loader, val_loader, test_loader
    del train_ds, val_ds, test_ds, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    return None, scalers, result

## 12. RBF-SVM Statistical Branch

SVM không học chuỗi thời gian, nhưng phù hợp với statistical vector cố định.

Pipeline:

```text
VarianceThreshold -> StandardScaler -> PCA -> RBF-SVM(probability=True)
```

RBF kernel:

\[
K(x_i, x_j) = \exp(-\gamma ||x_i - x_j||^2)
\]


In [ ]:
def fit_stats_svm(split_map):
    train_idx, val_idx, test_idx = split_map["train"], split_map["validation"], split_map["test"]
    start = time.time()
    n_components = min(STATS_PCA_COMPONENTS, max(2, len(train_idx) - 1), X_stats.shape[1])
    model = make_pipeline(
        VarianceThreshold(),
        StandardScaler(),
        PCA(n_components=n_components, random_state=SEED),
        SVC(kernel="rbf", C=8.0, gamma="scale", probability=True, class_weight="balanced", random_state=SEED),
    )
    X_train_stats = X_stats[train_idx]
    y_train_stats = y[train_idx]
    if X_stats_wave_aug is not None:
        aug_x = X_stats_wave_aug[:, train_idx, :].reshape(-1, X_stats.shape[1])
        aug_y = np.tile(y[train_idx], X_stats_wave_aug.shape[0])
        X_train_stats = np.vstack([X_train_stats, aug_x]).astype(np.float32)
        y_train_stats = np.concatenate([y_train_stats, aug_y]).astype(np.int64)
    model.fit(X_train_stats, y_train_stats)
    train_time = time.time() - start
    def predict(indices):
        t0 = time.time()
        prob = model.predict_proba(X_stats[indices])
        elapsed = time.time() - t0
        pred = prob.argmax(axis=1)
        yy = y[indices]
        return {
            "accuracy": accuracy_score(yy, pred),
            "macro_f1": f1_score(yy, pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(yy, pred, average="weighted", zero_division=0),
            "uar": balanced_accuracy_score(yy, pred),
            "macro_precision": precision_score(yy, pred, average="macro", zero_division=0),
            "macro_recall": recall_score(yy, pred, average="macro", zero_division=0),
            "y_true": yy,
            "y_pred": pred,
            "prob": prob,
            "elapsed_sec": elapsed,
        }
    return model, {"val": predict(val_idx), "test": predict(test_idx), "train_time_sec": train_time}


## Protocol Builder

Mặc định notebook này chỉ chạy 2 protocol chính để đúng kế hoạch 6 configs x 2 protocol:

```text
combined_random
combined_strict_no_tess
```

Có thể bật `RUN_SINGLE_DATASET=1` sau khi đã chọn được config tốt nhất.

In [ ]:
def split_map_from_column(column, include_mask=None):
    df = metadata if include_mask is None else metadata[include_mask].copy()
    return {
        "train": df.index[df[column].eq("train")].to_numpy(),
        "validation": df.index[df[column].eq("validation")].to_numpy(),
        "test": df.index[df[column].eq("test")].to_numpy(),
    }

protocols = []
if RUN_COMBINED_RANDOM:
    protocols.append(("combined_random", split_map_from_column("split_random")))
if RUN_COMBINED_STRICT:
    strict_mask = metadata["strict_include"] & metadata["split_strict_project"].isin(["train", "validation", "test"])
    protocols.append(("combined_strict_no_tess", split_map_from_column("split_strict_project", strict_mask)))

if RUN_SINGLE_DATASET:
    def random_single_dataset_split(dataset_name):
        idx = metadata.index[metadata["dataset"].eq(dataset_name)].to_numpy()
        labels = metadata.loc[idx, "label_id"]
        tr, temp = train_test_split(idx, test_size=0.30, random_state=SEED, stratify=labels)
        va, te = train_test_split(temp, test_size=0.50, random_state=SEED, stratify=metadata.loc[temp, "label_id"])
        return {"train": tr, "validation": va, "test": te}
    for ds in sorted(metadata["dataset"].unique()):
        protocols.append((f"single_{ds}", random_single_dataset_split(ds)))

print("Protocols:")
for name, smap in protocols:
    print(name, {k: len(v) for k, v in smap.items()})

print("Total deep runs:", len(protocols) * len(ACTIVE_CONFIGS))

## Run 6 Configs x 2 Protocols

Mỗi protocol sẽ fit một RBF-SVM statistical branch dùng chung, sau đó train 6 deep configs. Với mỗi config, notebook lưu cả deep metrics và stacking metrics.

In [ ]:
def metrics_row(protocol, config_name, model_name, split_name, result, n_samples):
    uar = result.get("uar")
    if uar is None and "y_true" in result and "y_pred" in result:
        uar = balanced_accuracy_score(result["y_true"], result["y_pred"])
    if uar is None:
        uar = result.get("macro_recall", np.nan)
    return {
        "protocol": protocol,
        "config_name": config_name,
        "model": model_name,
        "split": split_name,
        "n_samples": n_samples,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"],
        "weighted_f1": result["weighted_f1"],
        "uar": uar,
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "domain_accuracy": result.get("domain_accuracy", np.nan),
        "speaker_accuracy": result.get("speaker_accuracy", np.nan),
        "inference_time_sec": result.get("elapsed_sec", 0.0),
        "inference_ms_per_sample": 1000 * result.get("elapsed_sec", 0.0) / max(1, n_samples),
    }

metrics_path = REPORT_DIR / "06B_ablation_metrics.csv"
history_path = REPORT_DIR / "06B_ablation_history.csv"
RESUME_RUN_OUTPUTS = os.getenv("RESUME_RUN_OUTPUTS", "1") == "1"

def flush_progress():
    pd.DataFrame(all_metrics).to_csv(metrics_path, index=False)
    pd.DataFrame(all_history).to_csv(history_path, index=False)
    print("Checkpoint saved:", metrics_path, history_path)

if RESUME_RUN_OUTPUTS and metrics_path.exists():
    previous_metrics_df = pd.read_csv(metrics_path)
    all_metrics = previous_metrics_df.to_dict("records")
    completed_deep = set(
        zip(
            previous_metrics_df.loc[previous_metrics_df["model"].eq("deep_multibranch"), "protocol"],
            previous_metrics_df.loc[previous_metrics_df["model"].eq("deep_multibranch"), "config_name"],
        )
    )
    print("Loaded previous metrics:", metrics_path, "completed deep runs:", completed_deep)
else:
    all_metrics = []
    completed_deep = set()

if RESUME_RUN_OUTPUTS and history_path.exists():
    all_history = pd.read_csv(history_path).to_dict("records")
else:
    all_history = []

svm_cache = {}

for protocol_name, split_map in protocols:
    print("\n" + "=" * 100)
    print("PROTOCOL:", protocol_name, {k: len(v) for k, v in split_map.items()})
    print("=" * 100)
    train_idx, val_idx, test_idx = split_map["train"], split_map["validation"], split_map["test"]

    if RUN_SVM_BASELINE:
        svm_model, svm_result = fit_stats_svm(split_map)
        svm_cache[protocol_name] = svm_result
        has_svm_row = any(
            r.get("protocol") == protocol_name and r.get("config_name") == "shared" and r.get("model") == "rich_stats_rbf_svm"
            for r in all_metrics
        )
        if not has_svm_row:
            svm_row = metrics_row(protocol_name, "shared", "rich_stats_rbf_svm", "test", svm_result["test"], len(test_idx))
            svm_row.update({"best_epoch": 0, "best_val_macro_f1": svm_result["val"]["macro_f1"], "train_time_sec": svm_result["train_time_sec"]})
            all_metrics.append(svm_row)
            flush_progress()
        del svm_model
        gc.collect()
    else:
        svm_result = None

    for config_name in ACTIVE_CONFIGS:
        if (protocol_name, config_name) in completed_deep:
            print("SKIP completed deep run:", protocol_name, "/", config_name)
            continue

        print("\n" + "-" * 100)
        print("RUN:", protocol_name, "/", config_name)
        print(EXPERIMENT_CONFIGS[config_name]["note"])
        print("-" * 100)

        deep_model, scalers, deep_result = train_ablation_model(protocol_name, config_name, split_map)
        for row in deep_result["history"]:
            all_history.append(row)

        deep_row = metrics_row(protocol_name, config_name, "deep_multibranch", "test", deep_result["test"], len(test_idx))
        deep_row.update({
            "best_epoch": deep_result["best_epoch"],
            "best_val_macro_f1": deep_result["best_val_macro_f1"],
            "train_time_sec": deep_result["train_time_sec"],
            "note": EXPERIMENT_CONFIGS[config_name]["note"],
        })
        all_metrics.append(deep_row)

        if RUN_SVM_BASELINE:
            svm_result = svm_cache[protocol_name]
            X_meta_val = np.concatenate([deep_result["val"]["prob"], svm_result["val"]["prob"]], axis=1)
            X_meta_test = np.concatenate([deep_result["test"]["prob"], svm_result["test"]["prob"]], axis=1)
            meta = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
            meta_start = time.time()
            meta.fit(X_meta_val, y[val_idx])
            meta_train_time = time.time() - meta_start
            t0 = time.time()
            prob = meta.predict_proba(X_meta_test)
            elapsed = time.time() - t0
            pred = prob.argmax(axis=1)
            yy = y[test_idx]
            stack_result = {
                "accuracy": accuracy_score(yy, pred),
                "macro_f1": f1_score(yy, pred, average="macro", zero_division=0),
                "weighted_f1": f1_score(yy, pred, average="weighted", zero_division=0),
                "uar": balanced_accuracy_score(yy, pred),
                "macro_precision": precision_score(yy, pred, average="macro", zero_division=0),
                "macro_recall": recall_score(yy, pred, average="macro", zero_division=0),
                "domain_accuracy": np.nan,
                "speaker_accuracy": np.nan,
                "y_true": yy,
                "y_pred": pred,
                "prob": prob,
                "elapsed_sec": elapsed,
            }
            stack_row = metrics_row(protocol_name, config_name, "stacking_deep_plus_svm", "test", stack_result, len(test_idx))
            stack_row.update({"best_epoch": 0, "best_val_macro_f1": np.nan, "train_time_sec": meta_train_time})
            all_metrics.append(stack_row)

            pred_df = pd.DataFrame({
                "index": test_idx,
                "sample_id": metadata.loc[test_idx, "sample_id"].to_numpy(),
                "dataset": metadata.loc[test_idx, "dataset"].to_numpy(),
                "speaker_id": metadata.loc[test_idx, "speaker_id"].to_numpy(),
                "true_label": [ID_TO_LABEL[int(v)] for v in yy],
                "pred_label": [ID_TO_LABEL[int(v)] for v in pred],
                "protocol": protocol_name,
                "config_name": config_name,
            })
            for j, label in ID_TO_LABEL.items():
                pred_df[f"prob_{label}"] = prob[:, j]
            pred_df.to_csv(PRED_DIR / f"predictions_{protocol_name}_{config_name}_stacking.csv", index=False)

        completed_deep.add((protocol_name, config_name))
        flush_progress()
        for _name in [
            "deep_model", "scalers", "deep_result", "X_meta_val", "X_meta_test",
            "meta", "prob", "pred", "yy", "stack_result", "pred_df"
        ]:
            globals().pop(_name, None)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

metrics_df = pd.DataFrame(all_metrics)
history_df = pd.DataFrame(all_history)
metrics_df.to_csv(metrics_path, index=False)
history_df.to_csv(history_path, index=False)

display(metrics_df.sort_values(["protocol", "config_name", "model"]))
print("Saved:", metrics_path)
print("Saved:", history_path)

## Visualizations and Diagnostics

Các hình quan trọng:

- Macro-F1 theo config/protocol.
- Domain accuracy và speaker accuracy để đọc leakage.
- Bảng metric gồm accuracy, macro-F1, UAR, domain accuracy, speaker accuracy.

In [ ]:
if len(metrics_df):
    deep_stack_df = metrics_df[metrics_df["model"].isin(["deep_multibranch", "stacking_deep_plus_svm"])].copy()
    plt.figure(figsize=(15, 7))
    sns.barplot(data=deep_stack_df, x="config_name", y="macro_f1", hue="protocol")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.title("06B Ablation Macro-F1: 6 Configs x 2 Protocols")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "06B_ablation_macro_f1_by_config_protocol.png", dpi=180)
    plt.show()

    plt.figure(figsize=(15, 6))
    leak_df = metrics_df[metrics_df["model"].eq("deep_multibranch")].copy()
    leak_long = leak_df.melt(
        id_vars=["protocol", "config_name"],
        value_vars=["domain_accuracy", "speaker_accuracy"],
        var_name="leakage_metric",
        value_name="diagnostic_accuracy"
    )
    sns.barplot(data=leak_long, x="config_name", y="diagnostic_accuracy", hue="leakage_metric")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.title("Domain/Speaker Head Accuracy Diagnostics")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "06B_domain_speaker_leakage_diagnostics.png", dpi=180)
    plt.show()

    summary_cols = [
        "protocol", "config_name", "model", "accuracy", "macro_f1", "uar",
        "domain_accuracy", "speaker_accuracy", "best_epoch", "best_val_macro_f1",
        "inference_ms_per_sample"
    ]
    display(metrics_df[summary_cols].sort_values(["protocol", "macro_f1"], ascending=[True, False]))

    best_stack = metrics_df[metrics_df["model"].eq("stacking_deep_plus_svm")]
    if len(best_stack):
        best_rows = best_stack.loc[best_stack.groupby("protocol")["macro_f1"].idxmax()]
        display(best_rows[summary_cols])

## 16. Reference Comparison Table

Các paper reference thường báo kết quả single-dataset/random split, không chắc speaker-aware. Vì vậy notebook lưu bảng này như **reported reference**, không coi là so sánh tuyệt đối công bằng.


In [ ]:
reference_rows = [
    {
        "model": "Ahmed et al. weighted ensemble 1D-CNN + CNN-LSTM + CNN-GRU",
        "protocol": "single-dataset, split not clearly strict speaker-aware",
        "reported_accuracy_text": "TESS 99.46%; RAVDESS 95.62%; SAVEE 93.22%; CREMA-D 90.47%",
        "main_idea": "Handcrafted features, augmentation, recurrent CNN variants, weighted ensemble.",
        "link": "https://arxiv.org/abs/2112.05666",
    },
    {
        "model": "Chowdhury et al. DCRF-BiLSTM feature engineering",
        "protocol": "individual and combined datasets, not clearly same as our strict split",
        "reported_accuracy_text": "RAVDESS 97.83%; SAVEE 97.02%; CREMA-D 95.10%; TESS 100%; R+T+S+E+C 93.76%",
        "main_idea": "Richer handcrafted acoustic features, augmentation, BiLSTM temporal modeling, DeepCRF.",
        "link": "https://arxiv.org/abs/2507.07046",
    },
    {
        "model": "Ullah et al. 1D-CNN feature fusion",
        "protocol": "combined 4-dataset, split details not fully reproducible from local materials",
        "reported_accuracy_text": "CREMA-D + RAVDESS + SAVEE + TESS: 92.62%",
        "main_idea": "ZCR + energy + entropy of energy + RMS + MFCC -> 1D-CNN.",
        "link": "https://doi.org/10.1109/ICIT56493.2022.9989197",
    },
    {
        "model": "DANN / Gradient Reversal Layer",
        "protocol": "domain adaptation method, not SER-specific accuracy",
        "reported_accuracy_text": "Used as regularization method, not direct SER benchmark.",
        "main_idea": "Learn task-discriminative but domain-invariant representation.",
        "link": "https://arxiv.org/abs/1505.07818",
    },
]
ref_df = pd.DataFrame(reference_rows)
ref_df.to_csv(REPORT_DIR / "advanced_reference_table.csv", index=False)
display(ref_df)


## Save Output Package

In [ ]:
summary = {
    "notebook": "06B_Domain_Speaker_Ablation_SER_Unified",
    "objective": "Unified 06B notebook for fresh run, resume from saved cache/report, or continue from extracted outputs.",
    "configs": EXPERIMENT_CONFIGS,
    "active_configs": ACTIVE_CONFIGS,
    "protocols": [name for name, _ in protocols],
    "output_dir": str(OUTPUT_DIR),
    "metrics_csv": str(REPORT_DIR / "06B_ablation_metrics.csv"),
    "history_csv": str(REPORT_DIR / "06B_ablation_history.csv"),
    "notes": "Compare B0/B1/S1/S2/D3/C1 separately. Domain/speaker head accuracy is a leakage diagnostic, not the main task metric.",
    "modes": {
        "fresh": "IMPORT_PREVIOUS_RUN_OUTPUTS=0, PART2_REQUIRE_RESTORED_CACHE=0",
        "resume_cache": "IMPORT_PREVIOUS_RUN_OUTPUTS=1, PART2_REQUIRE_RESTORED_CACHE=1",
        "continue_outputs": "IMPORT_PREVIOUS_RUN_OUTPUTS=1, PREVIOUS_RUN_OUTPUT_ROOT=/path/to/output",
    },
}
with open(REPORT_DIR / "06B_ablation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

import zipfile

light_zip_path = PROJECT_ROOT / "06B_Unified_outputs_light_no_cache.zip"
light_package_dirs = [REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR]
with zipfile.ZipFile(light_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in light_package_dirs:
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, file_path.relative_to(OUTPUT_DIR))

full_zip_path = PROJECT_ROOT / "06B_Unified_outputs_FULL_WITH_CACHE.zip"
full_package_dirs = [CACHE_DIR, REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR]
with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in full_package_dirs:
        if folder.exists():
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    try:
                        arcname = file_path.relative_to(OUTPUT_DIR)
                    except ValueError:
                        arcname = Path("restored_cache") / file_path.name
                    zf.write(file_path, arcname)

print("LIGHT ZIP without cache:", light_zip_path)
print("FULL ZIP with cache:", full_zip_path)
full_zip_path
